In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm
import os
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
class TimeIntervalEmbedding(nn.Module):
    # разница во времени между книгами
    def __init__(self, hidden_dim, max_interval=365):
        super().__init__()
        self.emb = nn.Embedding(max_interval + 1, hidden_dim, padding_idx=0)

    def forward(self, intervals):
        intervals = intervals.clamp(0, self.emb.num_embeddings - 1).long()
        return self.emb(intervals)

In [ ]:
class SASRecTiSASRec(nn.Module):
    # модель с временными интервалами и признаком повтора

    def __init__(self, cnt_item, max_seq_len=30, hidden_dim=64,
                 num_heads=2, num_layers=2, dropout=0.2,
                 cnt_authors=0, cnt_categories=0, max_interval=365):
        super().__init__()

        self.max_seq_len = max_seq_len
        self.hidden_dim = hidden_dim

        self.item_emb = nn.Embedding(cnt_item + 1, hidden_dim, padding_idx=0)
        self.pos_emb = nn.Embedding(max_seq_len, hidden_dim)
        self.time_interval_emb = TimeIntervalEmbedding(hidden_dim, max_interval)
        self.repeat_emb = nn.Embedding(2, hidden_dim)

        if cnt_authors > 0:
            self.author_emb = nn.Embedding(cnt_authors + 1, hidden_dim, padding_idx=0)
        if cnt_categories > 0:
            self.category_emb = nn.Embedding(cnt_categories + 1, hidden_dim, padding_idx=0)